In [1]:
# Delhi AQI 2023 Data Preprocessing Pipeline (FIXED VERSION)
# Step-by-step cleaning for 15+ Delhi monitoring stations
# Compatible with CPCB 2023 pollutant data (PM2.5, PM10, NO2, SO2, CO, O3)

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# === STEP 1: CONFIGURATION ===
CONFIG = {
    'FOLDER_PATH': './',  # Same folder as this notebook
    'MONTH_MAP': {'January':1, 'February':2, 'March':3, 'April':4,
                  'May':5, 'June':6, 'July':7, 'August':8,
                  'September':9, 'October':10, 'November':11, 'December':12}
}

# Station coordinates (Lat, Long) for Tableau map
STATION_COORDS = {
    # Original stations (already working)
    'Anand Vihar': (28.6469, 77.3164),
    'Dwarka': (28.5921, 77.0460),
    'RK Puram': (28.5644, 77.1903),
    'Punjabi Bagh': (28.6742, 77.1313),
    'Okhla': (28.5355, 77.2677),
    'Rohini': (28.7350, 77.1101),
    'Jahangirpuri': (28.7330, 77.1632),
    'Mundka': (28.6847, 77.0336),
    'Wazirpur': (28.7040, 77.1646),
    'Nehru Nagar': (28.5672, 77.2502),
    'Siri Fort': (28.5500, 77.2167),
    'Mandir Marg': (28.6368, 77.2023),
    'Shadipur': (28.6478, 77.1476),
    'IGI Airport': (28.5562, 77.1000),
    'Narela': (28.8543, 77.0926),

    # YOUR NEW STATIONS (NaN fixed!)
    'Ashok Vihar': (28.6888, 77.1749),
    'Lodhi Road Delhi IITM': (28.5908, 77.2264),
    'Najafgarh': (28.6092, 76.9798),
    'North Campus': (28.6890, 77.2050),
    'Patparganj': (28.6140, 77.2890),
    'Pusa': (28.6370, 77.1770)
}

print(f"✅ Config loaded: {len(STATION_COORDS)} stations mapped (ALL NaN fixed!)")

✅ Config loaded: 21 stations mapped (ALL NaN fixed!)


In [4]:
# === STEP 2: HELPER FUNCTIONS ===
def extract_station_name(filepath):
    """Extract station name from filename"""
    stem = Path(filepath).stem
    # Remove common prefixes/suffixes
    stem = stem.replace('AQI_daily_2023_', '').replace('DelhiDPCC_2023_', '')
    stem = stem.replace('DelhiDPCC', '').replace('_2023', '')

    # More aggressive cleaning for matching with STATION_COORDS
    stem = stem.replace('_', ' ')
    stem = stem.replace(' (1)', '')
    stem = stem.replace(' (2)', '')
    stem = stem.replace('Delhi DPCC', '')
    stem = stem.replace('Delhi IMD', '')
    stem = stem.replace('Delhi CPCB', '')
    stem = stem.replace('Phase-2', '')
    stem = stem.replace('Sector 8', '')
    stem = stem.replace('R K Puram', 'RK Puram')
    stem = stem.replace('Sirifort', 'Siri Fort')
    stem = stem.replace('North Campus DU', 'North Campus') # Adjusting based on common station name
    stem = stem.replace('Lodhi Road', 'Lodhi Road') # No change for this, but keeping it in mind.
    stem = stem.replace('Pusa', 'Pusa') # No change for this
    stem = stem.replace('Dwarka-', 'Dwarka') # Fix for Dwarka- mismatch

    return stem.strip()

def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date
    df_long['Date'] = pd.to_datetime(df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%m-%d')
    df_long = df_long.assign(year=2023).dropna(subset=['Date'], errors='coerce')

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)

print("✅ Helper functions loaded")

✅ Helper functions loaded


In [5]:
# === STEP 3: LOAD AND MERGE ALL FILES ===

# Redefine load_one_file to handle date conversion errors
def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date, coercing invalid dates to NaT
    df_long['Date'] = pd.to_datetime('2023-' + df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%Y-%m-%d', errors='coerce')
    df_long = df_long.dropna(subset=['Date'])

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)


files = list(Path(CONFIG['FOLDER_PATH']).glob('*.xlsx')) + list(Path(CONFIG['FOLDER_PATH']).glob('*.xls'))

print(f"📁 Found {len(files)} Excel files")

dfs = []
for f in files:
    df_one = load_one_file(f)
    print(f"  - {extract_station_name(f)}: {len(df_one)} rows")
    dfs.append(df_one)

# Combine all stations
df = pd.concat(dfs, ignore_index=True)
df = df.sort_values(['Station', 'Date']).reset_index(drop=True)

print(f"\n📊 Combined dataset shape: {df.shape}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")

📁 Found 0 Excel files


ValueError: No objects to concatenate

In [ ]:
# === STEP 4: MISSING VALUE IMPUTATION ===
print("\n🧹 Step 4: Missing Value Imputation")

# Check missing by station
missing_by_station = df.groupby('Station')['AQI'].apply(lambda x: x.isna().sum()).reset_index()
missing_by_station.columns = ['Station', 'MissingDays']
missing_by_station['AvailableDays'] = 365 - missing_by_station['MissingDays']
missing_by_station['Missing_%'] = (missing_by_station['MissingDays'] / 365 * 100).round(1)

print("Missing values per station:")
print(missing_by_station.sort_values('MissingDays', ascending=False).to_string(index=False))

# Apply imputation (per station)
def impute_group(s):
    s = s.ffill()  # Forward fill first
    s = s.interpolate(method='linear', limit_direction='both')  # Then interpolate
    return s.round(0)

df['AQI'] = df.groupby('Station')['AQI'].transform(impute_group).astype(int)

print(f"\n✅ Missing AQI before: {df['AQI'].isna().sum()}")
print(f"✅ Missing AQI after: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 5: OUTLIER DETECTION (FLAG, DON'T REMOVE) ===
print("\n🔍 Step 5: Outlier Detection")

# IQR method (per station)
IQR_LowerBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.25) - 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
IQR_UpperBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.75) + 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
Outlier_IQR = (df['AQI'] < IQR_LowerBound) | (df['AQI'] > IQR_UpperBound)

# Z-score method (per station)
ZScore = df.groupby('Station')['AQI'].transform(lambda s: round((s - s.mean()) / s.std(), 2))
Outlier_Zscore = np.abs(ZScore) > 3

# Combined flags
df['Outlier_IQR'] = Outlier_IQR.astype(int)
df['Outlier_Zscore'] = Outlier_Zscore.astype(int)
df['Outlier_Either'] = (Outlier_IQR | Outlier_Zscore).astype(int)
df['Outlier_Both'] = (Outlier_IQR & Outlier_Zscore).astype(int)
df['ZScore'] = ZScore
df['IQR_LowerBound'] = IQR_LowerBound
df['IQR_UpperBound'] = IQR_UpperBound

print(f"🚩 Flagged by IQR: {df['Outlier_IQR'].sum()}")
print(f"🚩 Flagged by Z-score: {df['Outlier_Zscore'].sum()}")
print(f"🚩 Flagged by BOTH: {df['Outlier_Both'].sum()}")

In [ ]:
# === STEP 6: FEATURE ENGINEERING ===
print("\n⚙️ Step 6: Feature Engineering")

# Holiday flag (2023 Delhi festivals)
HOLIDAY_DICT = {
    '2023-01-01': 'New Year', '2023-01-14': 'Lohri', '2023-01-26': 'Republic Day',
    '2023-03-07': 'Holika Dahan', '2023-03-08': 'Holi', '2023-10-24': 'Dussehra',
    '2023-11-12': 'Diwali', '2023-11-13': 'Diwali Day 2', '2023-11-14': 'Bhai Dooj',
    '2023-12-25': 'Christmas'
}
# Correctly create a pandas Series with DatetimeIndex for holiday mapping
holiday_map = pd.Series(list(HOLIDAY_DICT.values()), index=pd.to_datetime(list(HOLIDAY_DICT.keys())))
df['IsHoliday'] = df['Date'].isin(holiday_map.index).astype(int)
df['HolidayName'] = df['Date'].map(holiday_map).fillna('Regular Day')

# Delhi seasons
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    elif month in [6, 7, 8, 9]: return 'Monsoon'
    else: return 'Post-Monsoon'

df['Season'] = df['Date'].dt.month.apply(get_season)
df['SeasonOrdinal'] = df['Season'].map({'Winter':1, 'Summer':2, 'Monsoon':3, 'Post-Monsoon':4})

# Time features
df['MonthName'] = df['Date'].dt.month_name()
df['MonthNum'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['DayofWeek'] = df['Date'].dt.day_name()
df['WeekNum'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df['IsWinter'] = df['Date'].dt.month.isin([11, 12, 1, 2]).astype(int)

print("✅ Feature engineering complete!")
print("\nSeason AQI averages:")
print(df.groupby('Season')['AQI'].mean().round(1).sort_values(ascending=False))

In [ ]:
df.info()

In [ ]:
# === STEP 2: HELPER FUNCTIONS ===
def extract_station_name(filepath):
    """Extract station name from filename"""
    stem = Path(filepath).stem
    # Remove common prefixes/suffixes
    stem = stem.replace('AQI_daily_2023_', '').replace('DelhiDPCC_2023_', '')
    stem = stem.replace('DelhiDPCC', '').replace('_2023', '')

    # More aggressive cleaning for matching with STATION_COORDS
    stem = stem.replace('_', ' ')
    stem = stem.replace(' (1)', '')
    stem = stem.replace(' (2)', '')
    stem = stem.replace('Delhi DPCC', '')
    stem = stem.replace('Delhi IMD', '')
    stem = stem.replace('Delhi CPCB', '')
    stem = stem.replace('Phase-2', '')
    stem = stem.replace('Sector 8', '')
    stem = stem.replace('R K Puram', 'RK Puram')
    stem = stem.replace('Sirifort', 'Siri Fort')
    stem = stem.replace('North Campus DU', 'North Campus') # Adjusting based on common station name
    stem = stem.replace('Lodhi Road', 'Lodhi Road') # No change for this, but keeping it in mind.
    stem = stem.replace('Pusa', 'Pusa') # No change for this
    stem = stem.replace('Dwarka-', 'Dwarka') # Fix for Dwarka- mismatch

    return stem.strip()

def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date
    df_long['Date'] = pd.to_datetime(df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%m-%d')
    df_long = df_long.assign(year=2023).dropna(subset=['Date'], errors='coerce')

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)

print("✅ Helper functions loaded")

In [ ]:
# === STEP 3: LOAD AND MERGE ALL FILES ===

# Redefine load_one_file to handle date conversion errors
def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date, coercing invalid dates to NaT
    df_long['Date'] = pd.to_datetime('2023-' + df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%Y-%m-%d', errors='coerce')
    df_long = df_long.dropna(subset=['Date'])

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)


files = list(Path(CONFIG['FOLDER_PATH']).glob('*.xlsx')) + list(Path(CONFIG['FOLDER_PATH']).glob('*.xls'))

print(f"📁 Found {len(files)} Excel files")

dfs = []
for f in files:
    df_one = load_one_file(f)
    print(f"  - {extract_station_name(f)}: {len(df_one)} rows")
    dfs.append(df_one)

# Combine all stations
df = pd.concat(dfs, ignore_index=True)
df = df.sort_values(['Station', 'Date']).reset_index(drop=True)

print(f"\n📊 Combined dataset shape: {df.shape}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 4: MISSING VALUE IMPUTATION ===
print("\n🧹 Step 4: Missing Value Imputation")

# Check missing by station
missing_by_station = df.groupby('Station')['AQI'].apply(lambda x: x.isna().sum()).reset_index()
missing_by_station.columns = ['Station', 'MissingDays']
missing_by_station['AvailableDays'] = 365 - missing_by_station['MissingDays']
missing_by_station['Missing_%'] = (missing_by_station['MissingDays'] / 365 * 100).round(1)

print("Missing values per station:")
print(missing_by_station.sort_values('MissingDays', ascending=False).to_string(index=False))

# Apply imputation (per station)
def impute_group(s):
    s = s.ffill()  # Forward fill first
    s = s.interpolate(method='linear', limit_direction='both')  # Then interpolate
    return s.round(0)

df['AQI'] = df.groupby('Station')['AQI'].transform(impute_group).astype(int)

print(f"\n✅ Missing AQI before: {df['AQI'].isna().sum()}")
print(f"✅ Missing AQI after: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 5: OUTLIER DETECTION (FLAG, DON'T REMOVE) ===
print("\n🔍 Step 5: Outlier Detection")

# IQR method (per station)
IQR_LowerBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.25) - 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
IQR_UpperBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.75) + 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
Outlier_IQR = (df['AQI'] < IQR_LowerBound) | (df['AQI'] > IQR_UpperBound)

# Z-score method (per station)
ZScore = df.groupby('Station')['AQI'].transform(lambda s: round((s - s.mean()) / s.std(), 2))
Outlier_Zscore = np.abs(ZScore) > 3

# Combined flags
df['Outlier_IQR'] = Outlier_IQR.astype(int)
df['Outlier_Zscore'] = Outlier_Zscore.astype(int)
df['Outlier_Either'] = (Outlier_IQR | Outlier_Zscore).astype(int)
df['Outlier_Both'] = (Outlier_IQR & Outlier_Zscore).astype(int)
df['ZScore'] = ZScore
df['IQR_LowerBound'] = IQR_LowerBound
df['IQR_UpperBound'] = IQR_UpperBound

print(f"🚩 Flagged by IQR: {df['Outlier_IQR'].sum()}")
print(f"🚩 Flagged by Z-score: {df['Outlier_Zscore'].sum()}")
print(f"🚩 Flagged by BOTH: {df['Outlier_Both'].sum()}")

In [ ]:
# === STEP 6: FEATURE ENGINEERING ===
print("\n⚙️ Step 6: Feature Engineering")

# Holiday flag (2023 Delhi festivals)
HOLIDAY_DICT = {
    '2023-01-01': 'New Year', '2023-01-14': 'Lohri', '2023-01-26': 'Republic Day',
    '2023-03-07': 'Holika Dahan', '2023-03-08': 'Holi', '2023-10-24': 'Dussehra',
    '2023-11-12': 'Diwali', '2023-11-13': 'Diwali Day 2', '2023-11-14': 'Bhai Dooj',
    '2023-12-25': 'Christmas'
}
# Correctly create a pandas Series with DatetimeIndex for holiday mapping
holiday_map = pd.Series(list(HOLIDAY_DICT.values()), index=pd.to_datetime(list(HOLIDAY_DICT.keys())))
df['IsHoliday'] = df['Date'].isin(holiday_map.index).astype(int)
df['HolidayName'] = df['Date'].map(holiday_map).fillna('Regular Day')

# Delhi seasons
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    elif month in [6, 7, 8, 9]: return 'Monsoon'
    else: return 'Post-Monsoon'

df['Season'] = df['Date'].dt.month.apply(get_season)
df['SeasonOrdinal'] = df['Season'].map({'Winter':1, 'Summer':2, 'Monsoon':3, 'Post-Monsoon':4})

# Time features
df['MonthName'] = df['Date'].dt.month_name()
df['MonthNum'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['DayofWeek'] = df['Date'].dt.day_name()
df['WeekNum'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df['IsWinter'] = df['Date'].dt.month.isin([11, 12, 1, 2]).astype(int)

print("✅ Feature engineering complete!")
print("\nSeason AQI averages:")
print(df.groupby('Season')['AQI'].mean().round(1).sort_values(ascending=False))

In [ ]:
# === STEP 7: FINAL SUMMARY & EXPORT ===
print("\n📊 FINAL DATASET SUMMARY")
print(f"📈 Rows: {len(df)}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")
print(f"🎉 Holiday days: {df['IsHoliday'].sum()}")
print(f"🚩 Outliers: {df['Outlier_Either'].sum()}")
print("\nAQI Category breakdown:")
print(df['Asthma_Risk'].value_counts() if 'Asthma_Risk' in df.columns else "Add Asthma_Risk manually")
print("\nSeason breakdown:")
print(df['Season'].value_counts())

# Add Asthma_Risk for choropleth
df['Asthma_Risk'] = pd.cut(df['AQI'],
                           bins=[0, 50, 100, float('inf')],
                           labels=['Safe', 'Caution', 'Unsafe'])

# Save final CSV for Tableau
OUTPUT_FILE = 'delhi_aqi_2023_tableau_ready.csv'
df.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d')

print(f"\n💾 Saved: {OUTPUT_FILE}")
print("✅ Ready for Tableau! Use columns: Latitude, Longitude, AQI, Asthma_Risk, Station")
print("📍 Map: Latitude/Longitude + Asthma_Risk (color)")
print("📈 Forecast: Date + AQI")
print("🔧 Filters: Zone, Season, IsHoliday, MonthName")

In [ ]:
print(df[['Station', 'Latitude', 'Longitude']].drop_duplicates().to_string(index=False))

In [ ]:
print(df[['Station', 'Latitude', 'Longitude']].drop_duplicates().to_string(index=False))

In [ ]:
print(df[['Station', 'Latitude', 'Longitude']].drop_duplicates().to_string(index=False))

In [ ]:
# === STEP 1: CONFIGURATION ===
CONFIG = {
    'FOLDER_PATH': './',  # Same folder as this notebook
    'MONTH_MAP': {'January':1, 'February':2, 'March':3, 'April':4,
                  'May':5, 'June':6, 'July':7, 'August':8,
                  'September':9, 'October':10, 'November':11, 'December':12}
}

# Station coordinates (Lat, Long) for Tableau map
STATION_COORDS = {
    'Anand Vihar': (28.6469, 77.3164),
    'Dwarka': (28.5921, 77.0460),
    'RK Puram': (28.5644, 77.1903),
    'Punjabi Bagh': (28.6742, 77.1313),
    'Okhla': (28.5355, 77.2677),
    'Rohini': (28.7350, 77.1101),
    'Jahangirpuri': (28.7330, 77.1632),
    'Mundka': (28.6847, 77.0336),
    'Wazirpur': (28.7040, 77.1646),
    'Nehru Nagar': (28.5672, 77.2502),
    'Siri Fort': (28.5500, 77.2167),
    'Mandir Marg': (28.6368, 77.2023),
    'Shadipur': (28.6478, 77.1476),
    'IGI Airport': (28.5562, 77.1000),
    'Narela': (28.8543, 77.0926),
    'Ashok Vihar': (28.6888, 77.1749),
    'Lodhi Road Delhi IITM': (28.5908, 77.2264),
    'Najafgarh': (28.6092, 76.9798),
    'North Campus': (28.6890, 77.2050),
    'Patparganj': (28.6140, 77.2890),
    'Pusa': (28.6370, 77.1770)
}

print(f"✅ Config loaded: {len(STATION_COORDS)} stations mapped")

In [ ]:
# === STEP 2: HELPER FUNCTIONS ===
def extract_station_name(filepath):
    """Extract station name from filename"""
    stem = Path(filepath).stem
    # Remove common prefixes/suffixes
    stem = stem.replace('AQI_daily_2023_', '').replace('DelhiDPCC_2023_', '')
    stem = stem.replace('DelhiDPCC', '').replace('_2023', '')

    # More aggressive cleaning for matching with STATION_COORDS
    stem = stem.replace('_', ' ')
    stem = stem.replace(' (1)', '')
    stem = stem.replace(' (2)', '')
    stem = stem.replace('Delhi DPCC', '')
    stem = stem.replace('Delhi IMD', '')
    stem = stem.replace('Delhi CPCB', '')
    stem = stem.replace('Phase-2', '')
    stem = stem.replace('Sector 8', '')
    stem = stem.replace('R K Puram', 'RK Puram')
    stem = stem.replace('Sirifort', 'Siri Fort')
    stem = stem.replace('North Campus DU', 'North Campus') # Adjusting based on common station name
    stem = stem.replace('Lodhi Road', 'Lodhi Road') # No change for this, but keeping it in mind.
    stem = stem.replace('Pusa', 'Pusa') # No change for this
    stem = stem.replace('Dwarka-', 'Dwarka') # Fix for Dwarka- mismatch

    return stem.strip()

def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date
    df_long['Date'] = pd.to_datetime(df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%m-%d')
    df_long = df_long.assign(year=2023).dropna(subset=['Date'], errors='coerce')

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)

print("✅ Helper functions loaded")

In [ ]:
# === STEP 3: LOAD AND MERGE ALL FILES ===

# Redefine load_one_file to handle date conversion errors
def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date, coercing invalid dates to NaT
    df_long['Date'] = pd.to_datetime('2023-' + df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%Y-%m-%d', errors='coerce')
    df_long = df_long.dropna(subset=['Date'])

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    print(f"DEBUG IN LOAD_ONE_FILE: Station: '{station}', Coords from map: {coords}") # Added debug line
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)


files = list(Path(CONFIG['FOLDER_PATH']).glob('*.xlsx')) + list(Path(CONFIG['FOLDER_PATH']).glob('*.xls'))

print(f"📁 Found {len(files)} Excel files")

dfs = []
for f in files:
    df_one = load_one_file(f)
    print(f"  - {extract_station_name(f)}: {len(df_one)} rows")
    dfs.append(df_one)

# Combine all stations
df = pd.concat(dfs, ignore_index=True)
df = df.sort_values(['Station', 'Date']).reset_index(drop=True)

print(f"\n📊 Combined dataset shape: {df.shape}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 4: MISSING VALUE IMPUTATION ===
print("\n🧹 Step 4: Missing Value Imputation")

# Check missing by station
missing_by_station = df.groupby('Station')['AQI'].apply(lambda x: x.isna().sum()).reset_index()
missing_by_station.columns = ['Station', 'MissingDays']
missing_by_station['AvailableDays'] = 365 - missing_by_station['MissingDays']
missing_by_station['Missing_%'] = (missing_by_station['MissingDays'] / 365 * 100).round(1)

print("Missing values per station:")
print(missing_by_station.sort_values('MissingDays', ascending=False).to_string(index=False))

# Apply imputation (per station)
def impute_group(s):
    s = s.ffill()  # Forward fill first
    s = s.interpolate(method='linear', limit_direction='both')  # Then interpolate
    return s.round(0)

df['AQI'] = df.groupby('Station')['AQI'].transform(impute_group).astype(int)

print(f"\n✅ Missing AQI before: {df['AQI'].isna().sum()}")
print(f"✅ Missing AQI after: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 5: OUTLIER DETECTION (FLAG, DON'T REMOVE) ===
print("\n🔍 Step 5: Outlier Detection")

# IQR method (per station)
IQR_LowerBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.25) - 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
IQR_UpperBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.75) + 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
Outlier_IQR = (df['AQI'] < IQR_LowerBound) | (df['AQI'] > IQR_UpperBound)

# Z-score method (per station)
ZScore = df.groupby('Station')['AQI'].transform(lambda s: round((s - s.mean()) / s.std(), 2))
Outlier_Zscore = np.abs(ZScore) > 3

# Combined flags
df['Outlier_IQR'] = Outlier_IQR.astype(int)
df['Outlier_Zscore'] = Outlier_Zscore.astype(int)
df['Outlier_Either'] = (Outlier_IQR | Outlier_Zscore).astype(int)
df['Outlier_Both'] = (Outlier_IQR & Outlier_Zscore).astype(int)
df['ZScore'] = ZScore
df['IQR_LowerBound'] = IQR_LowerBound
df['IQR_UpperBound'] = IQR_UpperBound

print(f"🚩 Flagged by IQR: {df['Outlier_IQR'].sum()}")
print(f"🚩 Flagged by Z-score: {df['Outlier_Zscore'].sum()}")
print(f"🚩 Flagged by BOTH: {df['Outlier_Both'].sum()}")

In [ ]:
# === STEP 6: FEATURE ENGINEERING ===
print("\n⚙️ Step 6: Feature Engineering")

# Holiday flag (2023 Delhi festivals)
HOLIDAY_DICT = {
    '2023-01-01': 'New Year', '2023-01-14': 'Lohri', '2023-01-26': 'Republic Day',
    '2023-03-07': 'Holika Dahan', '2023-03-08': 'Holi', '2023-10-24': 'Dussehra',
    '2023-11-12': 'Diwali', '2023-11-13': 'Diwali Day 2', '2023-11-14': 'Bhai Dooj',
    '2023-12-25': 'Christmas'
}
# Correctly create a pandas Series with DatetimeIndex for holiday mapping
holiday_map = pd.Series(list(HOLIDAY_DICT.values()), index=pd.to_datetime(list(HOLIDAY_DICT.keys())))
df['IsHoliday'] = df['Date'].isin(holiday_map.index).astype(int)
df['HolidayName'] = df['Date'].map(holiday_map).fillna('Regular Day')

# Delhi seasons
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    else: return 'Post-Monsoon' # Fallback for other months

df['Season'] = df['Date'].dt.month.apply(get_season)
df['SeasonOrdinal'] = df['Season'].map({'Winter':1, 'Summer':2, 'Monsoon':3, 'Post-Monsoon':4})

# Time features
df['MonthName'] = df['Date'].dt.month_name()
df['MonthNum'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['DayofWeek'] = df['Date'].dt.day_name()
df['WeekNum'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df['IsWinter'] = df['Date'].dt.month.isin([11, 12, 1, 2]).astype(int)

print("✅ Feature engineering complete!")
print("\nSeason AQI averages:")
print(df.groupby('Season')['AQI'].mean().round(1).sort_values(ascending=False))

In [ ]:
# === STEP 7: FINAL SUMMARY & EXPORT ===
print("\n📊 FINAL DATASET SUMMARY")
print(f"📈 Rows: {len(df)}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")
print(f"🎉 Holiday days: {df['IsHoliday'].sum()}")
print(f"🚩 Outliers: {df['Outlier_Either'].sum()}")
print("\nAQI Category breakdown:")
print(df['Asthma_Risk'].value_counts() if 'Asthma_Risk' in df.columns else "Add Asthma_Risk manually")
print("\nSeason breakdown:")
print(df['Season'].value_counts())

# Add Asthma_Risk for choropleth
df['Asthma_Risk'] = pd.cut(df['AQI'],
                           bins=[0, 50, 100, float('inf')],
                           labels=['Safe', 'Caution', 'Unsafe'])

# Save final CSV for Tableau
OUTPUT_FILE = 'delhi_aqi_2023_tableau_ready.csv'
df.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d')

print(f"\n💾 Saved: {OUTPUT_FILE}")
print("✅ Ready for Tableau! Use columns: Latitude, Longitude, AQI, Asthma_Risk, Station")
print("📍 Map: Latitude/Longitude + Asthma_Risk (color)")
print("📈 Forecast: Date + AQI")
print("🔧 Filters: Zone, Season, IsHoliday, MonthName")

In [ ]:
print(df[['Station', 'Latitude', 'Longitude']].drop_duplicates().to_string(index=False))

In [ ]:
# === STEP 2: HELPER FUNCTIONS ===
def extract_station_name(filepath):
    """Extract station name from filename"""
    stem = Path(filepath).stem
    # Remove common prefixes/suffixes
    stem = stem.replace('AQI_daily_2023_', '').replace('DelhiDPCC_2023_', '')
    stem = stem.replace('DelhiDPCC', '').replace('_2023', '')

    # More aggressive cleaning for matching with STATION_COORDS
    stem = stem.replace('_', ' ')
    stem = stem.replace(' (1)', '')
    stem = stem.replace(' (2)', '')
    stem = stem.replace('Delhi DPCC', '')
    stem = stem.replace('Delhi IMD', '')
    stem = stem.replace('Delhi CPCB', '')
    stem = stem.replace('Phase-2', '')
    stem = stem.replace('Sector 8', '')
    stem = stem.replace('R K Puram', 'RK Puram')
    stem = stem.replace('Sirifort', 'Siri Fort')
    stem = stem.replace('North Campus DU', 'North Campus') # Adjusting based on common station name
    stem = stem.replace('Lodhi Road', 'Lodhi Road') # No change for this, but keeping it in mind.
    stem = stem.replace('Pusa', 'Pusa') # No change for this
    stem = stem.replace('Dwarka-', 'Dwarka') # Fix for Dwarka- mismatch

    return stem.strip()

def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date
    df_long['Date'] = pd.to_datetime(df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%m-%d')
    df_long = df_long.assign(year=2023).dropna(subset=['Date'], errors='coerce')

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)

print("✅ Helper functions loaded")

In [ ]:
# === STEP 3: LOAD AND MERGE ALL FILES ===

# Redefine load_one_file to handle date conversion errors
def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date, coercing invalid dates to NaT
    df_long['Date'] = pd.to_datetime('2023-' + df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%Y-%m-%d', errors='coerce')
    df_long = df_long.dropna(subset=['Date'])

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    print(f"DEBUG IN LOAD_ONE_FILE: Station: '{station}', Coords from map: {coords}") # Added debug line
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)


files = list(Path(CONFIG['FOLDER_PATH']).glob('*.xlsx')) + list(Path(CONFIG['FOLDER_PATH']).glob('*.xls'))

print(f"📁 Found {len(files)} Excel files")

dfs = []
for f in files:
    df_one = load_one_file(f)
    print(f"  - {extract_station_name(f)}: {len(df_one)} rows")
    dfs.append(df_one)

# Combine all stations
df = pd.concat(dfs, ignore_index=True)
df = df.sort_values(['Station', 'Date']).reset_index(drop=True)

print(f"\n📊 Combined dataset shape: {df.shape}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 4: MISSING VALUE IMPUTATION ===
print("\n🧹 Step 4: Missing Value Imputation")

# Check missing by station
missing_by_station = df.groupby('Station')['AQI'].apply(lambda x: x.isna().sum()).reset_index()
missing_by_station.columns = ['Station', 'MissingDays']
missing_by_station['AvailableDays'] = 365 - missing_by_station['MissingDays']
missing_by_station['Missing_%'] = (missing_by_station['MissingDays'] / 365 * 100).round(1)

print("Missing values per station:")
print(missing_by_station.sort_values('MissingDays', ascending=False).to_string(index=False))

# Apply imputation (per station)
def impute_group(s):
    s = s.ffill()  # Forward fill first
    s = s.interpolate(method='linear', limit_direction='both')  # Then interpolate
    return s.round(0)

df['AQI'] = df.groupby('Station')['AQI'].transform(impute_group).astype(int)

print(f"\n✅ Missing AQI before: {df['AQI'].isna().sum()}")
print(f"✅ Missing AQI after: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 5: OUTLIER DETECTION (FLAG, DON'T REMOVE) ===
print("\n🔍 Step 5: Outlier Detection")

# IQR method (per station)
IQR_LowerBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.25) - 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
IQR_UpperBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.75) + 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
Outlier_IQR = (df['AQI'] < IQR_LowerBound) | (df['AQI'] > IQR_UpperBound)

# Z-score method (per station)
ZScore = df.groupby('Station')['AQI'].transform(lambda s: round((s - s.mean()) / s.std(), 2))
Outlier_Zscore = np.abs(ZScore) > 3

# Combined flags
df['Outlier_IQR'] = Outlier_IQR.astype(int)
df['Outlier_Zscore'] = Outlier_Zscore.astype(int)
df['Outlier_Either'] = (Outlier_IQR | Outlier_Zscore).astype(int)
df['Outlier_Both'] = (Outlier_IQR & Outlier_Zscore).astype(int)
df['ZScore'] = ZScore
df['IQR_LowerBound'] = IQR_LowerBound
df['IQR_UpperBound'] = IQR_UpperBound

print(f"🚩 Flagged by IQR: {df['Outlier_IQR'].sum()}")
print(f"🚩 Flagged by Z-score: {df['Outlier_Zscore'].sum()}")
print(f"🚩 Flagged by BOTH: {df['Outlier_Both'].sum()}")

In [ ]:
# === STEP 6: FEATURE ENGINEERING ===
print("\n⚙️ Step 6: Feature Engineering")

# Holiday flag (2023 Delhi festivals)
HOLIDAY_DICT = {
    '2023-01-01': 'New Year', '2023-01-14': 'Lohri', '2023-01-26': 'Republic Day',
    '2023-03-07': 'Holika Dahan', '2023-03-08': 'Holi', '2023-10-24': 'Dussehra',
    '2023-11-12': 'Diwali', '2023-11-13': 'Diwali Day 2', '2023-11-14': 'Bhai Dooj',
    '2023-12-25': 'Christmas'
}
# Correctly create a pandas Series with DatetimeIndex for holiday mapping
holiday_map = pd.Series(list(HOLIDAY_DICT.values()), index=pd.to_datetime(list(HOLIDAY_DICT.keys())))
df['IsHoliday'] = df['Date'].isin(holiday_map.index).astype(int)
df['HolidayName'] = df['Date'].map(holiday_map).fillna('Regular Day')

# Delhi seasons
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    else: return 'Post-Monsoon' # Fallback for other months

df['Season'] = df['Date'].dt.month.apply(get_season)
df['SeasonOrdinal'] = df['Season'].map({'Winter':1, 'Summer':2, 'Monsoon':3, 'Post-Monsoon':4})

# Time features
df['MonthName'] = df['Date'].dt.month_name()
df['MonthNum'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['DayofWeek'] = df['Date'].dt.day_name()
df['WeekNum'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df['IsWinter'] = df['Date'].dt.month.isin([11, 12, 1, 2]).astype(int)

print("✅ Feature engineering complete!")
print("\nSeason AQI averages:")
print(df.groupby('Season')['AQI'].mean().round(1).sort_values(ascending=False))

In [ ]:
# === STEP 7: FINAL SUMMARY & EXPORT ===
print("\n📊 FINAL DATASET SUMMARY")
print(f"📈 Rows: {len(df)}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")
print(f"🎉 Holiday days: {df['IsHoliday'].sum()}")
print(f"🚩 Outliers: {df['Outlier_Either'].sum()}")
print("\nAQI Category breakdown:")
print(df['Asthma_Risk'].value_counts() if 'Asthma_Risk' in df.columns else "Add Asthma_Risk manually")
print("\nSeason breakdown:")
print(df['Season'].value_counts())

# Add Asthma_Risk for choropleth
df['Asthma_Risk'] = pd.cut(df['AQI'],
                           bins=[0, 50, 100, float('inf')],
                           labels=['Safe', 'Caution', 'Unsafe'])

# Save final CSV for Tableau
OUTPUT_FILE = 'delhi_aqi_2023_tableau_ready.csv'
df.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d')

print(f"\n💾 Saved: {OUTPUT_FILE}")
print("✅ Ready for Tableau! Use columns: Latitude, Longitude, AQI, Asthma_Risk, Station")
print("📍 Map: Latitude/Longitude + Asthma_Risk (color)")
print("📈 Forecast: Date + AQI")
print("🔧 Filters: Zone, Season, IsHoliday, MonthName")

In [ ]:
print(df[['Station', 'Latitude', 'Longitude']].drop_duplicates().to_string(index=False))

In [ ]:
print(df[['Station', 'Latitude', 'Longitude']].drop_duplicates().to_string(index=False))

In [ ]:
# === STEP 3: LOAD AND MERGE ALL FILES ===

# Redefine load_one_file to handle date conversion errors
def load_one_file(filepath):
    """Load one Excel file and convert to long format"""
    station = extract_station_name(filepath)

    # Read only rows 1-31 (skip summary block at row 33+)
    df = pd.read_excel(filepath, header=0, nrows=31, engine='openpyxl')

    # Melt wide (Day x Month) -> long (one row per day)
    df_long = df.melt(id_vars=['Day'], var_name='Month', value_name='AQI')
    df_long['MonthNum'] = df_long['Month'].map(CONFIG['MONTH_MAP'])

    # Create proper date, coercing invalid dates to NaT
    df_long['Date'] = pd.to_datetime('2023-' + df_long['MonthNum'].astype(str) + '-' +
                                     df_long['Day'].astype(str), format='%Y-%m-%d', errors='coerce')
    df_long = df_long.dropna(subset=['Date'])

    # Convert AQI to numeric (NA -> NaN)
    df_long['AQI'] = pd.to_numeric(df_long['AQI'], errors='coerce')
    df_long['Station'] = station

    # Add coordinates
    coords = STATION_COORDS.get(station, (None, None))
    df_long['Latitude'] = coords[0]
    df_long['Longitude'] = coords[1]

    return df_long[['Date', 'Station', 'Latitude', 'Longitude', 'AQI']].sort_values('Date').reset_index(drop=True)


files = list(Path(CONFIG['FOLDER_PATH']).glob('*.xlsx')) + list(Path(CONFIG['FOLDER_PATH']).glob('*.xls'))

print(f"📁 Found {len(files)} Excel files")

dfs = []
for f in files:
    df_one = load_one_file(f)
    print(f"  - {extract_station_name(f)}: {len(df_one)} rows")
    dfs.append(df_one)

# Combine all stations
df = pd.concat(dfs, ignore_index=True)
df = df.sort_values(['Station', 'Date']).reset_index(drop=True)

print(f"\n📊 Combined dataset shape: {df.shape}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 4: MISSING VALUE IMPUTATION ===
print("\n🧹 Step 4: Missing Value Imputation")

# Check missing by station
missing_by_station = df.groupby('Station')['AQI'].apply(lambda x: x.isna().sum()).reset_index()
missing_by_station.columns = ['Station', 'MissingDays']
missing_by_station['AvailableDays'] = 365 - missing_by_station['MissingDays']
missing_by_station['Missing_%'] = (missing_by_station['MissingDays'] / 365 * 100).round(1)

print("Missing values per station:")
print(missing_by_station.sort_values('MissingDays', ascending=False).to_string(index=False))

# Apply imputation (per station)
def impute_group(s):
    s = s.ffill()  # Forward fill first
    s = s.interpolate(method='linear', limit_direction='both')  # Then interpolate
    return s.round(0)

df['AQI'] = df.groupby('Station')['AQI'].transform(impute_group).astype(int)

print(f"\n✅ Missing AQI before: {df['AQI'].isna().sum()}")
print(f"✅ Missing AQI after: {df['AQI'].isna().sum()}")

In [ ]:
# === STEP 5: OUTLIER DETECTION (FLAG, DON'T REMOVE) ===
print("\n🔍 Step 5: Outlier Detection")

# IQR method (per station)
IQR_LowerBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.25) - 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
IQR_UpperBound = df.groupby('Station')['AQI'].transform(lambda s: round(s.quantile(0.75) + 1.5*(s.quantile(0.75)-s.quantile(0.25)), 1))
Outlier_IQR = (df['AQI'] < IQR_LowerBound) | (df['AQI'] > IQR_UpperBound)

# Z-score method (per station)
ZScore = df.groupby('Station')['AQI'].transform(lambda s: round((s - s.mean()) / s.std(), 2))
Outlier_Zscore = np.abs(ZScore) > 3

# Combined flags
df['Outlier_IQR'] = Outlier_IQR.astype(int)
df['Outlier_Zscore'] = Outlier_Zscore.astype(int)
df['Outlier_Either'] = (Outlier_IQR | Outlier_Zscore).astype(int)
df['Outlier_Both'] = (Outlier_IQR & Outlier_Zscore).astype(int)
df['ZScore'] = ZScore
df['IQR_LowerBound'] = IQR_LowerBound
df['IQR_UpperBound'] = IQR_UpperBound

print(f"🚩 Flagged by IQR: {df['Outlier_IQR'].sum()}")
print(f"🚩 Flagged by Z-score: {df['Outlier_Zscore'].sum()}")
print(f"🚩 Flagged by BOTH: {df['Outlier_Both'].sum()}")

In [ ]:
# === STEP 6: FEATURE ENGINEERING ===
print("\n⚙️ Step 6: Feature Engineering")

# Holiday flag (2023 Delhi festivals)
HOLIDAY_DICT = {
    '2023-01-01': 'New Year', '2023-01-14': 'Lohri', '2023-01-26': 'Republic Day',
    '2023-03-07': 'Holika Dahan', '2023-03-08': 'Holi', '2023-10-24': 'Dussehra',
    '2023-11-12': 'Diwali', '2023-11-13': 'Diwali Day 2', '2023-11-14': 'Bhai Dooj',
    '2023-12-25': 'Christmas'
}
# Correctly create a pandas Series with DatetimeIndex for holiday mapping
holiday_map = pd.Series(list(HOLIDAY_DICT.values()), index=pd.to_datetime(list(HOLIDAY_DICT.keys())))
df['IsHoliday'] = df['Date'].isin(holiday_map.index).astype(int)
df['HolidayName'] = df['Date'].map(holiday_map).fillna('Regular Day')

# Delhi seasons
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5]: return 'Summer'
    else: return 'Post-Monsoon' # Fallback for other months

df['Season'] = df['Date'].dt.month.apply(get_season)
df['SeasonOrdinal'] = df['Season'].map({'Winter':1, 'Summer':2, 'Monsoon':3, 'Post-Monsoon':4})

# Time features
df['MonthName'] = df['Date'].dt.month_name()
df['MonthNum'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['DayofWeek'] = df['Date'].dt.day_name()
df['WeekNum'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df['IsWinter'] = df['Date'].dt.month.isin([11, 12, 1, 2]).astype(int)

print("✅ Feature engineering complete!")
print("\nSeason AQI averages:")
print(df.groupby('Season')['AQI'].mean().round(1).sort_values(ascending=False))

In [ ]:
# === STEP 7: FINAL SUMMARY & EXPORT ===
print("\n📊 FINAL DATASET SUMMARY")
print(f"📈 Rows: {len(df)}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")
print(f"🎉 Holiday days: {df['IsHoliday'].sum()}")
print(f"🚩 Outliers: {df['Outlier_Either'].sum()}")
print("\nAQI Category breakdown:")
print(df['Asthma_Risk'].value_counts() if 'Asthma_Risk' in df.columns else "Add Asthma_Risk manually")
print("\nSeason breakdown:")
print(df['Season'].value_counts())

# Add Asthma_Risk for choropleth
df['Asthma_Risk'] = pd.cut(df['AQI'],
                           bins=[0, 50, 100, float('inf')],
                           labels=['Safe', 'Caution', 'Unsafe'])

# Save final CSV for Tableau
OUTPUT_FILE = 'delhi_aqi_2023_tableau_ready.csv'
df.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d')

print(f"\n💾 Saved: {OUTPUT_FILE}")
print("✅ Ready for Tableau! Use columns: Latitude, Longitude, AQI, Asthma_Risk, Station")
print("📍 Map: Latitude/Longitude + Asthma_Risk (color)")
print("📈 Forecast: Date + AQI")
print("🔧 Filters: Zone, Season, IsHoliday, MonthName")

In [ ]:
# === STEP 7: FINAL SUMMARY & EXPORT ===
print("\n📊 FINAL DATASET SUMMARY")
print(f"📈 Rows: {len(df)}")
print(f"📍 Stations: {df['Station'].nunique()}")
print(f"📅 Date range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"❌ Missing AQI: {df['AQI'].isna().sum()}")
print(f"🎉 Holiday days: {df['IsHoliday'].sum()}")
print(f"🚩 Outliers: {df['Outlier_Either'].sum()}")
print("\nAQI Category breakdown:")
print(df['Asthma_Risk'].value_counts() if 'Asthma_Risk' in df.columns else "Add Asthma_Risk manually")
print("\nSeason breakdown:")
print(df['Season'].value_counts())

# Add Asthma_Risk for choropleth
df['Asthma_Risk'] = pd.cut(df['AQI'],
                           bins=[0, 50, 100, float('inf')],
                           labels=['Safe', 'Caution', 'Unsafe'])

# Save final CSV for Tableau
OUTPUT_FILE = 'delhi_aqi_2023_tableau_ready.csv'
df.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d')

print(f"\n💾 Saved: {OUTPUT_FILE}")
print("✅ Ready for Tableau! Use columns: Latitude, Longitude, AQI, Asthma_Risk, Station")
print("📍 Map: Latitude/Longitude + Asthma_Risk (color)")
print("📈 Forecast: Date + AQI")
print("🔧 Filters: Zone, Season, IsHoliday, MonthName")